# Предсказание |ΔS_M| и Tc по составу и данным об изготовлении

Ноутбук загружает две сохранённые лучшие модели (CatBoost на Stoichiometry-признаках) и **по очереди** спрашивает с клавиатуры:

1. формулу состава (например, `La0.7Ca0.3MnO3`);
2. размер кристаллитов, нм (обязательно);
3. температуру финального отжига, °C (можно пропустить);
4. способ получения (можно пропустить);
5. магнитное поле, Тл: одно или несколько через пробел (для ΔS).

Затем выводит Tc (K) и |ΔS_M| при каждом заданном поле. Образцов можно вводить сколько угодно; в конце таблицу можно сохранить в Excel.

**Как запускать:** выполните первые три ячейки по порядку, потом ячейку «Ввод образцов». Вопросы появляются в строке ввода вверху окна (VS Code) или под ячейкой (Jupyter). `q` на любом вопросе завершает ввод.

## Настройки и пути

In [1]:
# Предсказание |ΔS_M| и Tc по составу и данным об изготовлении
#
# Ноутбук загружает две уже обученные модели (лучшие по OOF R²) и по очереди
# запрашивает с клавиатуры всё, что нужно для предсказания.

import json
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from pymatgen.core import Composition

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Пути к сохранённым моделям (best_overall_model.joblib)
# ---------------------------------------------------------------------------

DELTA_S_MODEL_PATH = Path(
    r"/Users/artemykagramanian/Desktop/ампермагнит/MCE + ML/ipynb and results FMM version/MCE_Three_Feature_Sets_GridSearch_results/models/best_overall_model.joblib"
)

TC_MODEL_PATH = Path(
    r"/Users/artemykagramanian/Desktop/ампермагнит/MCE + ML/ipynb and results FMM version/Tc_Three_Feature_Sets_GridSearch_results/models/best_overall_model.joblib"
)

# Куда сохранить таблицу предсказаний (по желанию, в конце сессии).
OUTPUT_EXCEL = Path("predictions.xlsx")

# Нормировка формулы: как при обучении, кислород приводится к O = 3.
OXYGEN_REFERENCE = 3.0
STOICHIOMETRY_DECIMALS = 12

# Допустимые способы получения (как в обучающих данных).
ROUTE_MENU = {
    "1": ("sol_gel", "золь-гель"),
    "2": ("solid_state", "твердофазный синтез"),
    "3": ("combustion", "горение"),
    "4": ("other_route", "другой способ"),
}

In [2]:
# ## Функции


def clean_formula(formula):
    """Приводит строку химической формулы к единому виду."""
    text = (
        str(formula)
        .strip()
        .replace(" ", "")
        .replace("−", "-")
        .replace("–", "-")
        .replace(",", ".")
    )
    return text or None


def parse_formula(formula):
    """Преобразует химическую формулу в словарь состава."""
    cleaned = clean_formula(formula)
    if cleaned is None:
        raise ValueError("пустая формула")
    try:
        composition = Composition(cleaned).get_el_amt_dict()
    except Exception as error:
        raise ValueError(f"не удалось разобрать формулу: {error}") from error
    result = {}
    for element, amount in composition.items():
        amount = float(amount)
        if not np.isfinite(amount) or amount <= 0:
            raise ValueError(f"некорректный коэффициент у {element}: {amount}")
        result[str(element)] = amount
    if not result:
        raise ValueError("пустой состав после разбора")
    return result


def normalize_composition_to_o3(composition, oxygen_reference=OXYGEN_REFERENCE):
    """Нормирует все коэффициенты к O = 3 (так же, как при обучении)."""
    oxygen_amount = float(composition.get("O", np.nan))
    if not np.isfinite(oxygen_amount) or oxygen_amount <= 0:
        raise ValueError(
            "в формуле нет кислорода; запишите её полностью, "
            "например La0.7Ca0.3MnO3"
        )
    scale_factor = float(oxygen_reference) / oxygen_amount
    normalized = {}
    for element, amount in composition.items():
        value = float(amount) * scale_factor
        nearest_integer = round(value)
        if np.isclose(
            value,
            nearest_integer,
            atol=10 ** (-STOICHIOMETRY_DECIMALS),
            rtol=0.0,
        ):
            value = float(nearest_integer)
        else:
            value = float(round(value, STOICHIOMETRY_DECIMALS))
        if value > 0:
            normalized[str(element)] = value
    normalized["O"] = float(oxygen_reference)
    return normalized


def format_formula(composition):
    """Читаемая нормированная формула (кислород последним)."""
    parts = []
    # Порядок элементов как в введённой формуле, кислород в конце.
    for element in [e for e in composition if e != "O"] + ["O"]:
        amount = composition[element]
        text = (
            str(int(round(amount)))
            if np.isclose(amount, round(amount))
            else f"{amount:.4g}"
        )
        if element != "O" and text == "1":
            text = ""
        parts.append(f"{element}{text}")
    return "".join(parts)


PHYSICAL_COLUMNS = {
    "crystallite_size_nm",
    "magnetic_field_T",
    "process_T_final_C",
}


class Predictor:
    """Обёртка над сохранённой моделью: сама узнаёт, какие входы ей нужны."""

    def __init__(self, title, unit, model_path):
        self.title = title
        self.unit = unit
        self.model_path = Path(model_path)
        self.model = joblib.load(self.model_path)

        pipeline = getattr(self.model, "regressor_", self.model)
        preprocessor = pipeline.named_steps["preprocessor"]
        self.columns = list(preprocessor.feature_names_in_)

        unsupported = [
            column for column in self.columns
            if not (
                column.startswith("comp_")
                or column.startswith("process_route_")
                or column in PHYSICAL_COLUMNS
            )
        ]
        if unsupported:
            raise ValueError(
                f"Модель «{title}» использует признаки, которые этот ноутбук "
                f"считать не умеет (например {unsupported[:3]}). Скорее всего, "
                "она обучена на XenonPy-дескрипторах; нужна модель на "
                "Stoichiometry."
            )

        self.elements = [
            column[len("comp_"):]
            for column in self.columns
            if column.startswith("comp_")
        ]
        self.route_columns = [
            column for column in self.columns
            if column.startswith("process_route_")
        ]
        self.needs_field = "magnetic_field_T" in self.columns
        self.uses_t_final = "process_T_final_C" in self.columns

        # Метаданные: качество модели на отложенных составах.
        self.metadata = {}
        metadata_path = self.model_path.with_name(
            "best_overall_model_metadata.json"
        )
        if metadata_path.exists():
            self.metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

        # Обучающая таблица (по желанию): диапазоны и покрытие элементов.
        self.train = None
        table_path = (
            self.model_path.parent.parent / "tables" / "ml_dataset_all_features.csv"
        )
        if table_path.exists():
            self.train = pd.read_csv(table_path)
            if "composition_key" in self.train.columns:
                self.train_compositions = self.train.drop_duplicates(
                    "composition_key"
                )
            else:
                self.train_compositions = self.train

    def describe(self):
        meta = self.metadata
        lines = [f"{self.title}: {self.model_path.name}"]
        if meta:
            lines.append(
                f"  модель: {meta.get('model')} | набор признаков: "
                f"{meta.get('feature_set')} | OOF R² = "
                f"{meta.get('outer_oof_r2', float('nan')):.3f} | OOF MAE = "
                f"{meta.get('outer_oof_mae', float('nan')):.3g} {self.unit}"
            )
        lines.append(f"  входов модели: {len(self.columns)}")
        return "\n".join(lines)

    def typical_error(self):
        value = self.metadata.get("outer_oof_mae")
        return None if value is None else float(value)

    def warnings_for(self, composition, size_nm, field_values):
        messages = []
        for element in composition:
            if element == "O":
                continue
            if element not in self.elements:
                messages.append(
                    f"элемент {element} не встречался в обучающих данных, "
                    "модель его не учитывает: результат ненадёжен"
                )
            elif self.train is not None:
                column = f"comp_{element}"
                if column in self.train_compositions.columns:
                    count = int((self.train_compositions[column] > 0).sum())
                    if count < 3:
                        messages.append(
                            f"элемент {element} есть всего в {count} "
                            "обучающих составах: результат ненадёжен"
                        )
        if "Mn" not in composition:
            messages.append("в формуле нет Mn: модель обучена на манганитах")
        if self.train is not None:
            low, high = (
                float(self.train["crystallite_size_nm"].min()),
                float(self.train["crystallite_size_nm"].max()),
            )
            if not low <= size_nm <= high:
                messages.append(
                    f"размер кристаллитов {size_nm:g} нм вне диапазона "
                    f"обучения ({low:.3g}–{high:.3g} нм): экстраполяция"
                )
            if self.needs_field and "magnetic_field_T" in self.train.columns:
                low, high = (
                    float(self.train["magnetic_field_T"].min()),
                    float(self.train["magnetic_field_T"].max()),
                )
                for field in field_values:
                    if not low <= field <= high:
                        messages.append(
                            f"поле {field:g} Тл вне диапазона обучения "
                            f"({low:g}–{high:g} Тл): экстраполяция"
                        )
        return messages

    def make_frame(self, composition, size_nm, t_final_c, route, field_values):
        """Собирает входную таблицу с теми же столбцами, что при обучении."""
        rows = []
        for field in (field_values if self.needs_field else [np.nan]):
            row = {}
            for column in self.columns:
                if column.startswith("comp_"):
                    row[column] = float(composition.get(column[len("comp_"):], 0.0))
                elif column == "crystallite_size_nm":
                    row[column] = float(size_nm)
                elif column == "magnetic_field_T":
                    row[column] = float(field)
                elif column == "process_T_final_C":
                    row[column] = np.nan if t_final_c is None else float(t_final_c)
                elif column.startswith("process_route_"):
                    row[column] = float(column == f"process_route_{route}")
            rows.append(row)
        return pd.DataFrame(rows, columns=self.columns)

    def predict(self, composition, size_nm, t_final_c, route, field_values):
        frame = self.make_frame(composition, size_nm, t_final_c, route, field_values)
        return np.maximum(np.asarray(self.model.predict(frame), dtype=float), 0.0)


def load_predictor(title, unit, path):
    """Загружает модель; если не получилось, объясняет почему."""
    try:
        predictor = Predictor(title, unit, path)
    except Exception as error:
        print(f"[!] Модель «{title}» не загружена: {error}")
        return None
    print(predictor.describe())
    return predictor

## Загрузка моделей

In [3]:
delta_s_model = load_predictor("ΔS (|ΔS_M|)", "Дж/(кг·К)", DELTA_S_MODEL_PATH)
tc_model = load_predictor("Tc", "K", TC_MODEL_PATH)

ΔS (|ΔS_M|): best_overall_model.joblib
  модель: CatBoost | набор признаков: Stoichiometry | OOF R² = 0.626 | OOF MAE = 0.787 Дж/(кг·К)
  входов модели: 35
Tc: best_overall_model.joblib
  модель: CatBoost | набор признаков: Stoichiometry | OOF R² = 0.749 | OOF MAE = 27.9 K
  входов модели: 34


## Ввод образцов

In [4]:
# ## Интерактивный ввод


def parse_number(text, name, positive=True):
    """Число из строки; допускает запятую как десятичный знак и единицы в конце."""
    cleaned = str(text).strip().replace(",", ".")
    match = re.match(r"^[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", cleaned)
    if match is None:
        raise ValueError(f"{name}: не удалось прочитать число из «{text}»")
    if re.search(r"[\d.]", cleaned[match.end():]):
        raise ValueError(
            f"{name}: «{text}» не похоже на одно число "
            "(несколько значений разделяйте пробелом)"
        )
    value = float(match.group(0))
    if positive and value <= 0:
        raise ValueError(f"{name} должно быть больше нуля")
    return value


def ask(prompt, parser, optional=False):
    """Спрашивает, пока ответ не будет корректным. Пустой ответ = None, если optional."""
    while True:
        answer = input(prompt).strip()
        if answer.lower() in {"q", "й", "quit", "exit"}:
            raise KeyboardInterrupt
        if not answer:
            if optional:
                return None
            print("  Нужно ввести значение (q — выйти).")
            continue
        try:
            return parser(answer)
        except ValueError as error:
            print(f"  [!] {error}")


def parse_fields(text):
    """Поля через пробел или «;»; запятая считается десятичной (2,5 = 2.5 Тл)."""
    values = [
        parse_number(part, "поле")
        for part in re.split(r"[;\s]+", text.strip())
        if part
    ]
    if not values:
        raise ValueError("не указано ни одного значения поля")
    return sorted(set(values))


def ask_formula():
    def parser(text):
        composition = normalize_composition_to_o3(parse_formula(text))
        return composition
    return ask(
        "Формула состава, например La0.7Ca0.3MnO3 (q — закончить): ",
        parser,
    )


def ask_route(any_route_model):
    if not any_route_model:
        return "unknown"
    print("Способ получения:")
    for key, (_, label) in ROUTE_MENU.items():
        print(f"  {key} — {label}")
    print("  Enter — не знаю / не указан")

    def parser(text):
        if text not in ROUTE_MENU:
            raise ValueError("введите 1, 2, 3, 4 или нажмите Enter")
        return ROUTE_MENU[text][0]

    answer = ask("Способ получения (1–4 или Enter): ", parser, optional=True)
    return "unknown" if answer is None else answer


def run_interactive(delta_s_model=None, tc_model=None):
    """Запрашивает данные образца по очереди и печатает предсказания."""
    models = [m for m in (delta_s_model, tc_model) if m is not None]
    if not models:
        raise RuntimeError("Ни одна модель не загружена: проверьте пути в настройках.")

    history = []
    print("Ввод по очереди. На любой вопрос можно ответить q, чтобы закончить.\n")

    while True:
        try:
            composition = ask_formula()
            formula_o3 = format_formula(composition)
            print(f"  нормированная формула (O = 3): {formula_o3}")

            size_nm = ask("Размер кристаллитов, нм: ", lambda t: parse_number(t, "размер"))

            t_final = None
            if any(m.uses_t_final for m in models):
                t_final = ask(
                    "Температура финального отжига, °C (Enter — неизвестна): ",
                    lambda t: parse_number(t, "температура"),
                    optional=True,
                )

            route = ask_route(any(m.route_columns for m in models))

            fields = []
            if delta_s_model is not None and delta_s_model.needs_field:
                fields = ask(
                    "Магнитное поле, Тл (одно или несколько через пробел, "
                    "например 1 2 3 5): ",
                    parse_fields,
                )

            print("\n" + "-" * 64)
            print(f"Образец: {formula_o3}")
            print(
                f"  D = {size_nm:g} нм; T отжига = "
                f"{'неизвестна' if t_final is None else f'{t_final:g} °C'}; "
                f"способ: "
                f"{dict((v[0], v[1]) for v in ROUTE_MENU.values()).get(route, 'не указан')}"
            )

            record = {
                "Formula_O3": formula_o3,
                "Crystallite_size_nm": size_nm,
                "T_final_C": t_final,
                "Route": route,
            }

            if tc_model is not None:
                for message in tc_model.warnings_for(composition, size_nm, []):
                    print(f"  [!] {message}")
                tc = float(tc_model.predict(composition, size_nm, t_final, route, [])[0])
                error = tc_model.typical_error()
                tail = "" if error is None else f"   (типичная ошибка MAE ≈ {error:.3g} K)"
                print(f"\nTc = {tc:.0f} K{tail}")
                record["Tc_K"] = round(tc, 1)

            if delta_s_model is not None and fields:
                for message in delta_s_model.warnings_for(composition, size_nm, fields):
                    print(f"  [!] {message}")
                values = delta_s_model.predict(
                    composition, size_nm, t_final, route, fields
                )
                error = delta_s_model.typical_error()
                tail = "" if error is None else f"   (типичная ошибка MAE ≈ {error:.2g})"
                print(f"\n|ΔS_M|, Дж/(кг·К){tail}")
                for field, value in zip(fields, values):
                    print(f"  {field:g} Тл: {value:.2f}")
                    history.append({**record, "Field_T": field, "dS_J_kgK": round(float(value), 3)})
            else:
                history.append(record)

            print("-" * 64 + "\n")

        except KeyboardInterrupt:
            print("\nВвод завершён.")
            break

    table = pd.DataFrame(history)
    if not table.empty:
        display(table)
        answer = input(f"Сохранить таблицу в {OUTPUT_EXCEL}? (Enter — да, n — нет): ").strip().lower()
        if answer not in {"n", "н", "no"}:
            table.to_excel(OUTPUT_EXCEL, index=False)
            print("Сохранено:", Path(OUTPUT_EXCEL).resolve())
    return table

In [7]:
predictions = run_interactive(delta_s_model, tc_model)

Ввод по очереди. На любой вопрос можно ответить q, чтобы закончить.

  нормированная формула (O = 3): La0.7Ca0.25Sr0.05MnO3
Способ получения:
  1 — золь-гель
  2 — твердофазный синтез
  3 — горение
  4 — другой способ
  Enter — не знаю / не указан

----------------------------------------------------------------
Образец: La0.7Ca0.25Sr0.05MnO3
  D = 50 нм; T отжига = 700 °C; способ: горение

Tc = 277 K   (типичная ошибка MAE ≈ 27.9 K)

|ΔS_M|, Дж/(кг·К)   (типичная ошибка MAE ≈ 0.79)
  1 Тл: 0.62
  2 Тл: 1.27
  3 Тл: 2.09
  4 Тл: 2.54
  5 Тл: 3.29
----------------------------------------------------------------


Ввод завершён.


,Formula_O3,Crystallite_size_nm,T_final_C,Route,Tc_K,Field_T,dS_J_kgK
0,La0.7Ca0.25Sr0.05MnO3,50.0,700.0,combustion,277.4,1.0,0.620
1,La0.7Ca0.25Sr0.05MnO3,50.0,700.0,combustion,277.4,2.0,1.266
2,La0.7Ca0.25Sr0.05MnO3,50.0,700.0,combustion,277.4,3.0,2.087
3,La0.7Ca0.25Sr0.05MnO3,50.0,700.0,combustion,277.4,4.0,2.536
4,La0.7Ca0.25Sr0.05MnO3,50.0,700.0,combustion,277.4,5.0,3.291


Сохранено: /Users/artemykagramanian/Desktop/ампермагнит/MCE + ML/ipynb and results FMM version/predictions.xlsx


## Как читать результат и ограничения

- Обе модели обучены на **манганитах** (формулы вида A₁₋ₓA'ₓMnO₃ и их замещения в A- и B-позициях). Для других классов материалов предсказание не имеет смысла.
- «Типичная ошибка» в скобках — это MAE модели на составах, которые она не видела (внешняя кросс-валидация), а не доверительный интервал конкретного предсказания.
- Модели обучены на составах из статей. Для нового источника данных реальная ошибка может быть заметно больше (по нашему пилоту, при разбиении по статьям R² для Tc падает с ≈0,75 до ≈0,5).
- Предупреждения `[!]` показывают, когда вход выходит за диапазон обучающих данных или содержит редкий элемент.
- Если температура отжига или способ получения неизвестны, модель подставляет значения «не указано», как в обучающих данных.